# M4: Query and Generation Milestone

This notebook implements the final milestone: query processing and answer generation using the knowledge graph.

## Pipeline Overview

1. **Query → Candidate Concepts**: Extract key concepts from user query
2. **Concept → Subgraph**: Build semantically coherent subgraphs with:
   - Prerequisite chains (for scaffolded explanations)
   - Sibling nodes (for near-transfer)
   - Resource nodes (with spans and references)
3. **Subgraph → Answer**: Generate structured answer using LLM
4. **Provide References**: List all nodes and resources used

## Required Test Questions

1. **Attention in Transformers**: Explain dot-product self-attention, Q/K/V vectors, with Python example
2. **CLIP**: Explain text/image encoders, contrastive loss, one-shot classification
3. **Variational Lower Bound**: Explain Jensen's inequality, variational methods, VAE connection

## Setup and Imports

In [1]:
import sys
sys.path.append('/workspace')
from ingestion.mongo_helper import MongoHelper
import requests
import json
import networkx as nx
from collections import defaultdict
from typing import List, Dict, Set, Tuple
import re

print("Imports successful")

Imports successful


## Load Knowledge Graph from M3

In [2]:
# Initialize MongoDB and Ollama
mongo = MongoHelper()
OLLAMA_URL = "http://ollama:11434/api/generate"
MODEL = "qwen2.5:7b"

print("Connected to MongoDB")
print(f"Using Ollama model: {MODEL}")

# Load the graph
import pickle

with open('/workspace/knowledge_graph.pkl', 'rb') as f:
    graph = pickle.load(f)

print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

Connected to MongoDB
Using Ollama model: qwen2.5:7b
Graph loaded: 1573 nodes, 6783 edges


## Helper: Query Ollama

In [3]:
def query_ollama(prompt: str, temperature: float = 0.3, max_tokens: int = 2000) -> str:
    """Query the local Ollama LLM"""
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens
        }
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=120)
        if response.status_code == 200:
            return response.json()['response']
        else:
            print(f"Error: Status {response.status_code}")
            return None
    except Exception as e:
        print(f"Error querying Ollama: {e}")
        return None

## Step 1: Query → Candidate Concepts

Extract key AI/ML concepts from the user's query using the LLM.

In [4]:
def extract_query_concepts(query: str) -> List[str]:
    """
    Extract candidate concepts from user query.
    
    Args:
        query: User's question
    
    Returns:
        List of concept strings (normalized)
    """
    prompt = f"""Extract ALL relevant AI/ML concepts from this question, including both specific techniques and broader domains.

Question: {query}

Look for:
- Specific models and architectures (e.g., CLIP, ResNet, Transformer)
- Techniques and methods (e.g., attention mechanism, backpropagation)
- Mathematical concepts (e.g., Jensen's Inequality, ELBO)
- Loss functions and optimization (e.g., cross-entropy, Adam)
- Application domains (e.g., computer vision, NLP, reinforcement learning)
- Related concepts that help answer the question (e.g., if asking about CLIP, also include multimodal learning, vision-language models)

Return ONLY a JSON array of concept names. No explanations.

Examples:
Question: "How do transformers work in NLP?"
Answer: ["transformer", "attention mechanism", "self-attention", "NLP", "neural network"]

Question: "What is CLIP and how is it used in computer vision applications?"
Answer: ["CLIP", "computer vision", "vision-language model", "multimodal learning", "contrastive learning"]

Question: "Explain Jensen's Inequality in the context of ELBO"
Answer: ["Jensen's Inequality", "ELBO", "variational inference", "KL divergence"]

JSON array:"""
    
    response = query_ollama(prompt, temperature=0.2, max_tokens=200)
    
    if not response:
        return []
    
    try:
        # Clean response
        response = response.strip()
        # Remove markdown code blocks
        response = response.replace('```json', '').replace('```', '').strip()
        
        # Find JSON array
        start = response.find('[')
        end = response.rfind(']') + 1
        
        if start >= 0 and end > start:
            json_str = response[start:end]
            concepts = json.loads(json_str)
            
            # Normalize concept names to match graph node IDs
            normalized = [c.lower().replace(' ', '_').replace('-', '_') for c in concepts]
            
            print(f"Extracted concepts: {normalized}")  # Debug output
            return normalized
    except Exception as e:
        print(f"Error parsing concepts: {e}")
        print(f"Response was: {response[:300]}")
    
    return []

## Step 2: Concept → Subgraph

Build a semantically coherent subgraph around the candidate concepts.

**Strategy:**
1. Start with candidate concepts from query
2. Add prerequisite chains (traverse `prereq_of` edges backwards)
3. Add related concepts (siblings via `near_transfer`)
4. Add resource nodes that explain these concepts
5. Rank concepts by importance (based on query relevance + graph centrality)

In [5]:
def confirm_match_with_llm(candidate: str, node_title: str, node_definition: str) -> bool:
    """Ask the LLM whether a query candidate term genuinely refers to a
    matched graph concept, for cases where token-overlap scoring alone
    can't tell a harmless modifier ("python" + "programming") apart from
    a meaning-changing qualifier ("optimization" + "proximal policy")."""
    candidate_readable = candidate.replace('_', ' ')
    prompt = f"""You are matching a user's question to concepts in an AI course knowledge graph.

Query term: "{candidate_readable}"

Candidate matched concept: "{node_title}"
Definition: {node_definition or 'N/A'}

Question: Does the query term "{candidate_readable}" genuinely refer to the SAME concept as "{node_title}"? Answer "no" if the query term is more general, more specific, or refers to something meaningfully different (e.g. "optimization" does NOT refer to "Proximal Policy Optimization" specifically -- that's one narrow algorithm, not the general concept of optimization).

Answer with ONLY one word: "yes" or "no"
"""
    response = query_ollama(prompt, temperature=0.1)
    return bool(response and 'yes' in response.lower()[:20])


def find_matching_concepts(candidate_concepts: List[str], graph: nx.DiGraph) -> List[str]:
    """
    Find actual graph nodes that match candidate concepts.

    Scoring is proportional everywhere except true exact matches (node ID,
    alias, or title). Matches landing in a gray-zone score range (real
    overlap, but not certain) get a real LLM semantic check before being
    accepted, rather than another token-counting heuristic.
    """
    matching_nodes = []
    concept_nodes = [n for n, d in graph.nodes(data=True) if d.get('node_type') == 'concept']

    GRAY_ZONE_FLOOR = 30
    CONFIDENT_THRESHOLD = 100

    print(f"\n Matching {len(candidate_concepts)} candidates against {len(concept_nodes)} nodes...")

    for candidate in candidate_concepts:
        best_match = None
        best_score = 0
        candidate_tokens = set(candidate.split('_'))

        for node in concept_nodes:
            score = 0
            node_data = graph.nodes[node]
            node_tokens = set(node.split('_'))

            if candidate == node:
                best_match, best_score = node, 100
                break

            overlap = node_tokens & candidate_tokens
            if overlap:
                score = max(score, int(len(overlap) / max(len(node_tokens), len(candidate_tokens)) * 85))

            aliases = node_data.get('aliases', [])
            normalized_aliases = [a.lower().replace(' ', '_').replace('-', '_') for a in aliases]
            if candidate in normalized_aliases:
                best_match, best_score = node, 100
                break

            for alias in normalized_aliases:
                alias_tokens = set(alias.split('_'))
                overlap = alias_tokens & candidate_tokens
                if overlap:
                    score = max(score, int(len(overlap) / max(len(alias_tokens), len(candidate_tokens)) * 85))

            title = node_data.get('title', '')
            if title:
                normalized_title = title.lower().replace(' ', '_').replace('-', '_')
                if candidate == normalized_title:
                    score = 100
                else:
                    title_tokens = set(normalized_title.split('_'))
                    overlap = title_tokens & candidate_tokens
                    if overlap:
                        score = max(score, int(len(overlap) / max(len(title_tokens), len(candidate_tokens)) * 85))

            if score > best_score:
                best_score = score
                best_match = node

        if not best_match or best_score < GRAY_ZONE_FLOOR:
            print(f"    '{candidate}' - no match found")
            continue

        if best_score >= CONFIDENT_THRESHOLD:
            matching_nodes.append(best_match)
            print(f"    '{candidate}' → '{best_match}' (score: {best_score}, exact)")
            continue

        node_title = graph.nodes[best_match].get('title', best_match)
        node_def = graph.nodes[best_match].get('definition', '')
        if confirm_match_with_llm(candidate, node_title, node_def):
            matching_nodes.append(best_match)
            print(f"    '{candidate}' → '{best_match}' (score: {best_score}, LLM-confirmed)")
        else:
            print(f"    '{candidate}' - rejected '{best_match}' (score: {best_score}, LLM said different concept)")

    result = list(set(matching_nodes))
    print(f"\n Final matches: {result}\n")
    return result


def build_subgraph(query_concepts: List[str], graph: nx.DiGraph, max_depth: int = 2) -> Dict:
    """
    Build a subgraph around query concepts including prerequisites and related concepts.
    
    Args:
        query_concepts: List of concept node IDs from the query
        graph: Full knowledge graph
        max_depth: How many hops to include for prerequisites
    
    Returns:
        Dictionary with:
        - 'concepts': List of concept node IDs in the subgraph
        - 'resources': List of resource node IDs
        - 'prerequisite_chains': List of ordered concept chains (simple → complex)
        - 'related_concepts': List of sibling/related concepts
    """
    subgraph_concepts = set(query_concepts)
    resources = set()
    prerequisite_chains = []
    related_concepts = set()
    
    # 1. Add prerequisites (traverse backwards from query concepts)
    for concept in query_concepts:
        # Find prerequisite chain
        chain = [concept]
        current = concept
        
        # Traverse up to max_depth hops backwards
        for _ in range(max_depth):
            # Find edges where current concept is the target
            prereqs = [source for source, target, data in graph.in_edges(current, data=True)
                      if data.get('relation') == 'prereq_of' and 
                      graph.nodes[source].get('node_type') == 'concept']
            
            if prereqs:
                # Pick the first prerequisite (you could rank by difficulty)
                prereq = prereqs[0]
                chain.insert(0, prereq)  # Add to beginning (simpler concepts first)
                subgraph_concepts.add(prereq)
                current = prereq
            else:
                break
        
        if len(chain) > 1:
            prerequisite_chains.append(chain)
    
    # 2. Add related concepts (near-transfer)
    for concept in query_concepts:
        neighbors = [target for source, target, data in graph.out_edges(concept, data=True)
                    if data.get('relation') == 'near_transfer' and
                    graph.nodes[target].get('node_type') == 'concept']
        related_concepts.update(neighbors)
        subgraph_concepts.update(neighbors)
    
    # 3. Add all resources that explain any concept in the subgraph
    for concept in subgraph_concepts:
        # Find resources that explain this concept
        explaining_resources = [source for source, target, data in graph.in_edges(concept, data=True)
                               if data.get('relation') == 'explains' and
                               graph.nodes[source].get('node_type') == 'resource']
        resources.update(explaining_resources)

    # 4. Add worked examples that exemplify any concept in the subgraph
    examples = set()
    for concept in subgraph_concepts:
        exemplifying = [source for source, target, data in graph.in_edges(concept, data=True)
                        if data.get('relation') == 'exemplifies' and
                        graph.nodes[source].get('node_type') == 'example']
        examples.update(exemplifying)

    return {
        'concepts': list(subgraph_concepts),
        'resources': list(resources),
        'examples': list(examples),
        'prerequisite_chains': prerequisite_chains,
        'related_concepts': list(related_concepts)
    }

## Step 3: Subgraph → Answer Generation

Generate a structured answer using the subgraph context and LLM.

**System Prompt Design:**
- You are an AI tutor explaining concepts progressively (simple → complex)
- Use prerequisite chains to scaffold explanations
- Include code examples when requested
- Ground all explanations in provided resource content

In [6]:
# System prompt (used for all questions)
SYSTEM_PROMPT = """You are an AI tutor for an Introduction to AI course. Your role is to:

1. Explain concepts progressively from foundational to advanced
2. Use the prerequisite chain provided to scaffold your explanation
3. When code examples are requested, provide clear, well-commented Python code
4. Base all explanations on the course materials provided in the context
5. Be concise but thorough, focusing on understanding over memorization

Structure your answers as:
- Start with simpler prerequisite concepts
- Build up to the main concept
- Provide examples or code when requested
- Explain the reasoning and intuition, not just the mechanics
"""


def _fetch_resource_content(mongo: MongoHelper, resource_node: Dict) -> str:
    """Fetch resource content from the RIGHT collection based on resource
    type. Resources now come from three sources (webpages/slides/videos,
    per the spec's type in [pdf, slide, video, web]) -- looking up every
    resource in mongo.db.webpages only, regardless of its actual type,
    silently dropped all slide- and video-sourced resources from context."""
    url = resource_node.get('url', '')
    rtype = resource_node.get('type', 'web')
    if rtype == 'video':
        doc = mongo.db.videos.find_one({'url': url})
    elif rtype in ('pdf', 'slide'):
        doc = mongo.db.slides.find_one({'url': url})
    else:
        doc = mongo.db.webpages.find_one({'url': url})
    return doc['content'] if doc else None


def build_context_from_subgraph(subgraph: Dict, graph: nx.DiGraph, mongo: MongoHelper) -> str:
    """
    Build context string from subgraph for LLM prompt.
    
    Args:
        subgraph: Subgraph dictionary from build_subgraph()
        graph: Knowledge graph
        mongo: MongoDB helper to fetch resource content
    
    Returns:
        Context string for LLM
    """
    context_parts = []
    
    # 1. Prerequisite chains
    if subgraph['prerequisite_chains']:
        context_parts.append("PREREQUISITE CHAIN (simple → complex):")
        for chain in subgraph['prerequisite_chains']:
            chain_str = " → ".join([graph.nodes[c]['title'] for c in chain])
            context_parts.append(f"  {chain_str}")
        context_parts.append("")
    
    # 2. Concept definitions
    context_parts.append("CONCEPTS:")
    for concept_id in subgraph['concepts']:
        node = graph.nodes[concept_id]
        title = node.get('title', concept_id)
        definition = node.get('definition', 'No definition available')
        difficulty = node.get('difficulty', 'intermediate')
        
        context_parts.append(f"\n[{title}] ({difficulty})")
        context_parts.append(f"Definition: {definition}")

    # 2b. Related concepts (near_transfer) -- useful for the LLM to know
    # what's related/contrasting, even if not pulled in as full context
    if subgraph.get('related_concepts'):
        related_titles = [graph.nodes[c].get('title', c) for c in subgraph['related_concepts']
                          if c in graph.nodes]
        if related_titles:
            context_parts.append(f"\nRELATED CONCEPTS (for comprehension/comparison, not required background): {', '.join(related_titles)}")

    context_parts.append("")

    # 2c. Worked examples
    if subgraph.get('examples'):
        context_parts.append("WORKED EXAMPLES:")
        for example_id in subgraph['examples']:
            ex = graph.nodes[example_id]
            context_parts.append(f"  - {ex.get('title', 'Example')}: {ex.get('snippet', '')}")
        context_parts.append("")
    
    # 3. Resource content (sample from each resource)
    context_parts.append("COURSE MATERIALS:")
    for resource_id in subgraph['resources'][:10]:  # Limit to 10 resources to avoid context overflow
        resource = graph.nodes[resource_id]
        url = resource.get('url', '')
        span = resource.get('span', '')
        rtype = resource.get('type', 'web')
        
        content = _fetch_resource_content(mongo, resource)
        if content:
            content_sample = content[:1000]
            context_parts.append(f"\n[{resource.get('title', 'Resource')}]")
            context_parts.append(f"URL: {url}")
            if rtype == 'video':
                context_parts.append(f"Timecode: {span}")
            else:
                context_parts.append(f"Span: {span}")
            context_parts.append(f"Content: {content_sample}...")
    
    return "\n".join(context_parts)


def generate_answer(query: str, subgraph: Dict, graph: nx.DiGraph, mongo: MongoHelper) -> Dict:
    """
    Generate answer using LLM with subgraph context.
    
    Args:
        query: User's question
        subgraph: Subgraph dictionary
        graph: Knowledge graph
        mongo: MongoDB helper
    
    Returns:
        Dictionary with answer, nodes used, and resources
    """
    # Build context
    context = build_context_from_subgraph(subgraph, graph, mongo)
    
    # Build full prompt
    full_prompt = f"""{SYSTEM_PROMPT}

CONTEXT FROM COURSE MATERIALS:
{context}

STUDENT QUESTION:
{query}

YOUR ANSWER:"""
    
    # Generate answer
    answer = query_ollama(full_prompt, temperature=0.5, max_tokens=2000)
    
    # Prepare response with metadata
    return {
        'answer': answer,
        'nodes_used': {
            'concepts': [{
                'id': c,
                'title': graph.nodes[c].get('title', c),
                'difficulty': graph.nodes[c].get('difficulty', 'intermediate')
            } for c in subgraph['concepts']],
            'resources': [{
                'id': r,
                'url': graph.nodes[r].get('url', ''),
                'span': graph.nodes[r].get('span', ''),
                'type': graph.nodes[r].get('type', 'web'),
                'title': graph.nodes[r].get('title', '')
            } for r in subgraph['resources']],
            'examples': [{
                'id': e,
                'title': graph.nodes[e].get('title', ''),
                'snippet': graph.nodes[e].get('snippet', '')
            } for e in subgraph.get('examples', [])],
            'related_concepts': [{
                'id': c,
                'title': graph.nodes[c].get('title', c)
            } for c in subgraph.get('related_concepts', [])]
        },
        'prerequisite_chains': subgraph['prerequisite_chains'],
        'system_prompt': SYSTEM_PROMPT,
        'full_prompt': full_prompt
    }

## Complete Pipeline: Query → Answer

In [7]:
def process_query(query: str, graph: nx.DiGraph, mongo: MongoHelper) -> Dict:
    """
    Complete pipeline: query → concepts → subgraph → answer.
    
    Args:
        query: User's question
        graph: Knowledge graph
        mongo: MongoDB helper
    
    Returns:
        Dictionary with answer and metadata
    """
    print(f"\nProcessing query: {query}")
    print("=" * 70)
    
    # Step 1: Extract candidate concepts
    print("\n[Step 1] Extracting candidate concepts from query...")
    candidate_concepts = extract_query_concepts(query)
    print(f"Candidates: {candidate_concepts}")
    
    # Step 2: Find matching concepts in graph
    print("\n[Step 2] Finding matching concepts in knowledge graph...")
    query_concepts = find_matching_concepts(candidate_concepts, graph)
    print(f"Matched concepts: {query_concepts}")
    
    if not query_concepts:
        print("WARNING: No matching concepts found in graph!")
        return {
            'answer': ("I don't have course materials covering this topic well enough to answer "
                       "confidently. This may not be part of the ingested course content -- "
                       "you may want to check with your instructor or search external resources."),
            'nodes_used': {'concepts': [], 'resources': [], 'examples': [], 'related_concepts': []},
            'prerequisite_chains': [],
            'system_prompt': SYSTEM_PROMPT,
            'full_prompt': None
        }
    
    # Step 3: Build subgraph
    print("\n[Step 3] Building subgraph with prerequisites and related concepts...")
    subgraph = build_subgraph(query_concepts, graph, max_depth=2)
    print(f"Subgraph contains:")
    print(f"  - {len(subgraph['concepts'])} concepts")
    print(f"  - {len(subgraph['resources'])} resources")
    print(f"  - {len(subgraph['prerequisite_chains'])} prerequisite chains")
    
    # Step 4: Generate answer
    print("\n[Step 4] Generating answer with LLM...")
    result = generate_answer(query, subgraph, graph, mongo)
    
    return result


def display_result(result: Dict):
    """
    Display the result in a readable format.
    """
    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)
    print(result['answer'])
    
    print("\n" + "=" * 70)
    print("KNOWLEDGE GRAPH NODES USED")
    print("=" * 70)
    
    print("\nConcepts:")
    for concept in result['nodes_used']['concepts']:
        print(f"  - {concept['title']} ({concept['difficulty']})")
    
    print(f"\nResources ({len(result['nodes_used']['resources'])} total):")
    for resource in result['nodes_used']['resources'][:5]:  # Show first 5
        print(f"  - {resource['title']} [{resource.get('type', 'web')}]")
        print(f"    URL: {resource['url']}")
        if resource.get('type') == 'video':
            print(f"    Timecode: {resource['span']}")
        else:
            print(f"    Span: {resource['span']}")
    
    if len(result['nodes_used']['resources']) > 5:
        print(f"  ... and {len(result['nodes_used']['resources']) - 5} more resources")

    if result['nodes_used'].get('examples'):
        print(f"\nWorked Examples ({len(result['nodes_used']['examples'])} total):")
        for example in result['nodes_used']['examples']:
            print(f"  - {example['title']}: {example['snippet']}")

    if result['nodes_used'].get('related_concepts'):
        titles = [c['title'] for c in result['nodes_used']['related_concepts']]
        print(f"\nRelated Concepts (near_transfer, for comprehension checks): {', '.join(titles)}")
    
    if result.get('prerequisite_chains'):
        print("\nPrerequisite Chains Used:")
        for chain in result['prerequisite_chains']:
            print(f"  - Chain: {' → '.join(chain)}")
    
    print("\n" + "=" * 70)
    print("SYSTEM PROMPT USED")
    print("=" * 70)
    print(result['system_prompt'])

## Test Questions

### Question 1: Attention in Transformers

In [8]:
question_1 = "What is attention in transformers and can you provide a python example of how it is used?"

result_1 = process_query(question_1, graph, mongo)
display_result(result_1)


Processing query: What is attention in transformers and can you provide a python example of how it is used?

[Step 1] Extracting candidate concepts from query...
Extracted concepts: ['transformer', 'attention_mechanism', 'self_attention', 'nlp', 'neural_network', 'python_example', 'loss_functions', 'cross_entropy', 'optimization', 'computer_vision']
Candidates: ['transformer', 'attention_mechanism', 'self_attention', 'nlp', 'neural_network', 'python_example', 'loss_functions', 'cross_entropy', 'optimization', 'computer_vision']

[Step 2] Finding matching concepts in knowledge graph...

 Matching 10 candidates against 1026 nodes...
    'transformer' → 'transformer' (score: 100, exact)
    'attention_mechanism' → 'attention_mechanism' (score: 100, exact)
    'self_attention' → 'self_attention' (score: 100, exact)
    'nlp' - no match found
    'neural_network' → 'neural_networks' (score: 100, exact)
    'python_example' - no match found
    'loss_functions' → 'loss_functions' (score: 10

### Question 2: CLIP

In [9]:
question_2 = "What is CLIP and how it is used in computer vision applications?"

result_2 = process_query(question_2, graph, mongo)
display_result(result_2)


Processing query: What is CLIP and how it is used in computer vision applications?

[Step 1] Extracting candidate concepts from query...
Extracted concepts: ['clip', 'computer_vision', 'vision_language_model', 'multimodal_learning', 'contrastive_learning']
Candidates: ['clip', 'computer_vision', 'vision_language_model', 'multimodal_learning', 'contrastive_learning']

[Step 2] Finding matching concepts in knowledge graph...

 Matching 5 candidates against 1026 nodes...
    'clip' - rejected 'clip_model' (score: 42, LLM said different concept)
    'computer_vision' - rejected 'vision_transformers' (score: 42, LLM said different concept)
    'vision_language_model' - rejected 'vision_language_action_(vla)_models' (score: 34, LLM said different concept)
    'multimodal_learning' - rejected 'multimodal_distributions' (score: 42, LLM said different concept)
    'contrastive_learning' → 'contrastive_learning' (score: 100, exact)

 Final matches: ['contrastive_learning']

Matched concepts: ['

### Question 3: Variational Lower Bound

In [10]:
question_3 = "Can you explain the variational lower bound and how it relates to Jensen's inequality?"

result_3 = process_query(question_3, graph, mongo)
display_result(result_3)


Processing query: Can you explain the variational lower bound and how it relates to Jensen's inequality?

[Step 1] Extracting candidate concepts from query...
Extracted concepts: ['variational_lower_bound', "jensen's_inequality", 'variational_inference', 'kl_divergence']
Candidates: ['variational_lower_bound', "jensen's_inequality", 'variational_inference', 'kl_divergence']

[Step 2] Finding matching concepts in knowledge graph...

 Matching 4 candidates against 1026 nodes...
    'variational_lower_bound' → 'evidence_lower_bound_(elbo)' (score: 100, exact)
    'jensen's_inequality' → 'jensen's_inequality' (score: 100, exact)
    'variational_inference' → 'variational_inference' (score: 100, exact)
    'kl_divergence' → 'kl_divergence' (score: 100, exact)

 Final matches: ["jensen's_inequality", 'evidence_lower_bound_(elbo)', 'variational_inference', 'kl_divergence']

Matched concepts: ["jensen's_inequality", 'evidence_lower_bound_(elbo)', 'variational_inference', 'kl_divergence']

[St

In [11]:
# Check which results are valid
print("Checking results...")
for i, (question, result) in enumerate([
    (question_1, result_1),
    (question_2, result_2),
    (question_3, result_3)
], 1):
    print(f"\nQuestion {i}: {question[:50]}...")
    print(f"  Has 'full_prompt': {'full_prompt' in result}")
    print(f"  Has 'answer': {'answer' in result}")
    print(f"  Concepts used: {len(result.get('nodes_used', {}).get('concepts', []))}")
    
    if 'full_prompt' not in result:
        print(f"    WARNING: This result is incomplete!")
        print(f"  Available keys: {list(result.keys())}")

Checking results...

Question 1: What is attention in transformers and can you prov...
  Has 'full_prompt': True
  Has 'answer': True
  Concepts used: 20

Question 2: What is CLIP and how it is used in computer vision...
  Has 'full_prompt': True
  Has 'answer': True
  Concepts used: 2

Question 3: Can you explain the variational lower bound and ho...
  Has 'full_prompt': True
  Has 'answer': True
  Concepts used: 23


## Save Full Prompts for Documentation

Save the complete prompts used for each question (for M4 submission).

In [12]:
# Save prompts to file (with error handling)
with open('/workspace/m4_prompts.txt', 'w') as f:
    f.write("M4 QUERY AND GENERATION - PROMPTS USED\n")
    f.write("=" * 70 + "\n\n")
    
    for i, (question, result) in enumerate([
        (question_1, result_1),
        (question_2, result_2),
        (question_3, result_3)
    ], 1):
        f.write(f"QUESTION {i}:\n")
        f.write(f"{question}\n\n")
        
        # Check if full_prompt exists
        if 'full_prompt' in result:
            f.write(f"FULL PROMPT SENT TO LLM:\n")
            f.write("=" * 70 + "\n")
            f.write(result['full_prompt'])
            f.write("\n\n" + "=" * 70 + "\n\n")
        else:
            f.write(f"ERROR: No prompt generated\n")
            f.write(f"Answer: {result.get('answer', 'No answer available')}\n\n")
            f.write("=" * 70 + "\n\n")

print("Prompts saved to /workspace/m4_prompts.txt")

Prompts saved to /workspace/m4_prompts.txt


## Summary Statistics

In [13]:
print("\n" + "=" * 70)
print("M4 PIPELINE SUMMARY")
print("=" * 70)

print("\nSubgraph Retrieval Strategy:")
print("  1. Extract candidate concepts from query using LLM")
print("  2. Match candidates to graph nodes (exact + fuzzy + alias matching)")
print("  3. Build subgraph by adding:")
print("     - Prerequisite chains (backward traversal up to 2 hops)")
print("     - Related concepts (near-transfer edges)")
print("     - All resources explaining subgraph concepts")
print("  4. Generate answer from simple → complex using prerequisite order")

print("\nRationale:")
print("  - Prerequisites ensure foundational understanding before advanced topics")
print("  - Related concepts provide broader context and connections")
print("  - Resource spans give exact references for student verification")
print("  - Progressive explanation matches pedagogical best practices")

print("\nResults:")
for i, result in enumerate([result_1, result_2, result_3], 1):
    print(f"\n  Question {i}:")
    print(f"    - Concepts used: {len(result['nodes_used']['concepts'])}")
    print(f"    - Resources used: {len(result['nodes_used']['resources'])}")
    print(f"    - Prerequisite chains: {len(result.get('prerequisite_chains', []))}")


M4 PIPELINE SUMMARY

Subgraph Retrieval Strategy:
  1. Extract candidate concepts from query using LLM
  2. Match candidates to graph nodes (exact + fuzzy + alias matching)
  3. Build subgraph by adding:
     - Prerequisite chains (backward traversal up to 2 hops)
     - Related concepts (near-transfer edges)
     - All resources explaining subgraph concepts
  4. Generate answer from simple → complex using prerequisite order

Rationale:
  - Prerequisites ensure foundational understanding before advanced topics
  - Related concepts provide broader context and connections
  - Resource spans give exact references for student verification
  - Progressive explanation matches pedagogical best practices

Results:

  Question 1:
    - Concepts used: 20
    - Resources used: 48
    - Prerequisite chains: 1

  Question 2:
    - Concepts used: 2
    - Resources used: 3
    - Prerequisite chains: 0

  Question 3:
    - Concepts used: 23
    - Resources used: 16
    - Prerequisite chains: 3
